## Import Required Libraries

In [3]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Define Transformations

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


## Load Dataset

In [5]:
IMAGE_WIDTH = 672.0
DATA_DIR  = 'regression_dataset/data'
LABELS_CSV = 'regression_dataset/labels.csv'
BATCH_SIZE = 128
NUM_WORKERS  = 16
SEED = 42
NUM_EPOCHS = 200
MODEL_SAVE_PATH = 'regression_model_simple_baseline.pth'
PATIENCE = 15
MIN_DELTA = 1e-6

torch.manual_seed(SEED)

df = pd.read_csv(LABELS_CSV)
n_total = len(df)
n_train = int(0.8 * n_total)
n_val   = n_total - n_train

# Split indices first so train and val images get different transforms.
# Applying train_transform to all images before the split would leak
# ColorJitter augmentation into the validation set.
perm          = torch.randperm(n_total, generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices = perm[:n_train]
val_indices   = perm[n_train:]

# Load train images with augmentation transforms
train_images_list, train_labels_list = [], []
for i in train_indices:
    row = df.iloc[i]
    img_path = os.path.join(DATA_DIR, f"frame_{int(row['frame_id'])}.jpg")
    image = Image.open(img_path).convert('RGB')
    train_images_list.append(train_transform(image))
    train_labels_list.append(float(row['center_x']) / IMAGE_WIDTH)

# Load val images with clean val_transform
val_images_list, val_labels_list = [], []
for i in val_indices:
    row = df.iloc[i]
    img_path = os.path.join(DATA_DIR, f"frame_{int(row['frame_id'])}.jpg")
    image = Image.open(img_path).convert('RGB')
    val_images_list.append(val_transform(image))
    val_labels_list.append(float(row['center_x']) / IMAGE_WIDTH)

train_dataset = TensorDataset(
    torch.stack(train_images_list),
    torch.tensor(train_labels_list, dtype=torch.float32),
)
val_dataset = TensorDataset(
    torch.stack(val_images_list),
    torch.tensor(val_labels_list, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Total: {n_total}, Train: {n_train}, Val: {n_val}")


## Define Model Architecture

In [6]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, 1)

model = model.to(device)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001,
    weight_decay=5e-4
)

loss_fn = nn.SmoothL1Loss()


## Train the Model

In [7]:
best_val_mae_px   = float('inf')
epochs_no_improve = 0
train_mse_history = []
val_mse_history   = []

for epoch in range(1, NUM_EPOCHS + 1):
    # Training phase
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        preds = model(images)
        batch_loss = loss_fn(preds, labels)
        batch_loss.backward()
        optimizer.step()
        running_train_loss += batch_loss.item() * images.size(0)
    train_mse = running_train_loss / n_train

    # Validation phase
    model.eval()
    running_val_loss = 0.0
    running_val_mae  = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            preds  = model(images)
            running_val_loss += loss_fn(preds, labels).item() * images.size(0)
            running_val_mae  += torch.abs(preds - labels).sum().item()
    val_mse    = running_val_loss / n_val
    val_mae_px = (running_val_mae / n_val) * IMAGE_WIDTH

    train_mse_history.append(train_mse)
    val_mse_history.append(val_mse)

    if best_val_mae_px - val_mae_px > MIN_DELTA:
        best_val_mae_px   = val_mae_px
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
    else:
        epochs_no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}: Train MSE: {train_mse:.6f}, Val MSE: {val_mse:.6f}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs.")
        break

print(f"\nBest Val MAE: {best_val_mae_px:.2f} px")


## Evaluate the Model

In [8]:
# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(train_mse_history) + 1)
axes[0].plot(epochs_ran, train_mse_history, label='Train MSE')
axes[0].plot(epochs_ran, val_mse_history,   label='Val MSE')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE (normalised)')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Load best checkpoint for final evaluation
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()

all_preds  = []
all_labels = []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds  = model(images).squeeze(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()  * IMAGE_WIDTH
all_labels = torch.cat(all_labels).numpy() * IMAGE_WIDTH

mae_px = np.mean(np.abs(all_preds - all_labels))
print(f"Final Val MAE: {mae_px:.2f} px")

axes[1].hist(all_labels, bins=30, alpha=0.6, label='Ground truth')
axes[1].hist(all_preds,  bins=30, alpha=0.6, label='Predictions')
axes[1].set_xlabel('center_x (px)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Predictions vs Ground Truth')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [9]:
import shutil
import os
from datetime import datetime

DEST_DIR  = '/'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
dest_path = os.path.join(DEST_DIR, f'regression_model_baseline_{timestamp}.pth')

os.makedirs(DEST_DIR, exist_ok=True)
shutil.copy2(MODEL_SAVE_PATH, dest_path)

print(f'Model copied to {dest_path}')